# 03-thinking.ipynb

langgraph 로 agent 를 만들 때, 어떻게 생각하고 기획하는지. [참고문서](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph#draft-reply)

1. Node로 단계 쪼개기 + Edge로 연결하기
1. 각 Node의 상세 작성
1. 공유하는 State를 구성

## 고객 지원 이메일링 agent 기획 튜토리얼

Agent 요구사항 :

- 수신 고객 이메일을 읽음
- 급한 정도와 주제로 분류함
- 답변에 필요한 관련있는 문서를 검색
- 적절한 대응 초안 작성
- 복잡한 문제는 담당자에게 전달(Human In the Loop)
- 필요시 후속 조치 예약

예시 사용자 시나리오 :

1. 단순 제품 질문 "전원 껐다 키는 방법 뭐예요?"
2. 버그 리포트 : "pdf 추출했더니 꺼져요""
3. 긴급 결제 이슈 : "구독료 두 번 나감!"
4. 기능 추가 요청 : "모바일에서 다크모드 만들어주세요"
5. 복잡한 기술적 문제 : "API 연동이 가끔 504 오류 뜨면서 실패함"

## 구현에 필요한 5단계

1. 워크플로우를 개별 단계로 나눔
2. 각 단계에서 어떤 일을 처리할지 구체화
3. `State` 기획/디자인
4. 필요한 Node/Tool/Router 개발
5. 전체 조립

### 1. 워크플로우를 개별 단계로 나눔
- 그래프 디자인
```mermaid
flowchart TD

    START([START])
    READ[이메일 분석]
    CLASSIFY[의도 파악]

    DOC[문서 검색]
    BUG[버그 추적]
    HR1[사람 리뷰1]

    DRAFT[답장 초안]

    HR2[사람 리뷰2]
    SEND[답장 보내기]

    END([END])

    START --> READ
    READ --> CLASSIFY

    CLASSIFY -.-> DOC
    CLASSIFY -.-> BUG
    CLASSIFY -.-> HR1

    DOC --> DRAFT
    BUG --> DRAFT
    HR1 --> DRAFT

    DRAFT -.-> HR2
    DRAFT -.-> SEND

    HR2 --> END
    SEND --> END
```


### 2. 각 단계에서 어떤 일을 처리할지 구체화
> 단일 노드(함수) | 단일 권한 | 단일 책임
#### LLM 단계
- 의도 분류
    - Input (State) : 카테고리 분류, 급한 단계, 응답 포맷
    - Prompt : email 내용, 발신자 정보
    - Output : 구조화된 분류 -> 다음 스텝 결정에 활용
- 초안 작성
    - Input : 분류 결과, 검색 결과, 고객 기록
    - Prompt : 어투 안내, 회사 정책, 답변 템플릿
    - Output : 전문적인 답변 메일 내용

#### Data 단계
- 문서 검색
    - Parameter : 사용자 의도와 주제에 맞는 Query
    - Retry : 일시적 오류시, 0.5초-1초-2초-4초 후에 재시도
    - Caching : 자주 사용하는 Query와 결과는 캐싱 가능
- 고객 기록 확인
    - Parameter : 사용자 email / ID
    - Retry : 시도, 만약 정보를 불러올 수 없으면 기본 정보로 대체
    - Caching : 정보의 실시간성을 위해 캐싱 유통기한 설정


#### 액션(API) 단계
- 이메일 읽기
- 답장 보내기
    - When : 승인 이후 (사람/자동화)
    - Retry : 0.2, 0.4, 0.8, 1.6 초 마다 재시도
    - Return : Status Code(200, 400, 500)
- 버그 추적
    - When : 의도가 bug 로 파악된 경우 항상
    - Retry : 0.2, 0.4, 0.8, 1.6 초 마다 재시도
    - Return : 버그 이슈 티켓 ID

#### 사람 개입 단계
- 사람 리뷰
    - When : 매우 긴급, 복잡한 이슈, 답장 품질 우려
    - Context : 수신 이메일 원본, 답장 초안, 긴급 정도, 분류 카테고리
    - Human Input : O/X (보낸다 만다), 필요하다면 고친 내용

### 3. `State` 기획/디자인
> **어떤 정보를 `state` 에 저장할 것인가**
- 포함 : 각 단계에 영구적으로 필요한 데이터인가?
- 비포함 : 다른 데이터에서 가져올 수 있는가?

**Email 에이전트에서 추적해야 할 데이터** -> `state`
- `sender_email` : 발신자 이메일 주소
- `email_content` : 메일 내용

- `classification` : `intent`, `urgency`, `topic`, `summary`

- `search_results` : 문서 검색 결과
- `customer_history` : 사용자 상담 기록

- `draft_response` : 초안
- `messages` : agent가 step별로 생성한 모든 기록(메모리)


**중요한 원칙**
> `state` 는 날것의 데이터 (JSON, `dict`)를 그대로 사용. 자연어로 바꾸지 말 것.

In [3]:
state = {
    "sender_email" : "a@t.com",
    "email_content" : "버그 발생",
    "classification" : {
        "intent" : "question" | "bug" | "billing" | "feature" | "complex",
        "urgency" : "low" | "medium" | "high" | "critical",
        "topic" : "???",
        "summary" : "사용자 결제 단계에서 문제가 발생"
    },
    "search_results" : ["결제오류는 ~~~처리한다", "결제가 이뤄지지 ~~~~", "어리모어뢰ㅏㅓ"],
    "customer_history" : {},
    "draft_response" : "안녕하세요, 고객님. 불편~~~",
    "messages" : ["", "", "", ""]
}

TypeError: unsupported operand type(s) for |: 'str' and 'str'

In [ ]:
from typing import TypedDict, Literal, NotRequired

class EmailClassification(TypedDict):
    intent : Literal['question', 'bug', 'billing', 'feature', 'complex']
    urgency : Literal['low', 'medium', 'high', 'critical']
    topic : str
    summary : str

class EmailAgentState(TypedDict):
    sender_email : str
    email_content : str
    
    classification : EmailClassification

    search_results : list[str] | None   # str 으로 이뤄진 list 거나 None 이거나
    customer_history : dict | None

    draft_response : str | None
    messages : list[str] | None

In [ ]:
# State 정의
from typing import TypedDict, Literal, NotRequired

class TeacherState(TypedDict):
    user_input: str # 사용자 입력
    sentiment: NotRequired[Literal['positive', 'negative', 'aggressive']] # 감정 분석
    core_msg : NotRequired[str] # 핵심 메시지
    response : NotRequired[str] # 최종 답변
    

### 4. 필요한 Node/Tool/Router 개발

### 5. 전체 조립